# 04 — Strategies

**Workstream D** of the group project plan. This notebook builds the eight weight functions, each with a
stated objective, constraints and parameters. It tests them, and it runs the two checks the brief
asks for explicitly: whether **IV differs from ERC**, and whether **MV is distinct from GMV and MSR**.
It produces **no performance results**. Those come from notebook 05.

| Task | Output |
| --- | --- |
| D1–D8 | `strategies.STRATEGIES`: EW, GMV, MV, MSR, IV, ERC, MDP, MDC |
| D9 | One signature, `f(returns) -> weights`, interchangeable in `backtest.run_backtest`, with unit tests |
| D10 | **Table 3.2 — Strategy specifications**, including the shortcomings column the group's draft asks for |
| D11 | **Table 3.3 — IV against ERC** on one fixed window, plus the same comparison across all 400 rebalances |
| D12 | **Table 3.4 — MV, GMV and MSR** on the same window, plus a distinctness count across all 400 |

Shared code: `src/strategies.py`. The engine is `src/backtest.py` (notebook 03).

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.optimize import minimize

from src import backtest as bt
from src import config as cfg
from src import data, fmt
from src import strategies as st

np.random.seed(cfg.SEED)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

---

## Data and the 400 estimation windows

The panel and the 100 experiments are rebuilt exactly as in notebook 03. Each experiment rebalances
four times (quarterly), so a strategy is asked for weights on **400 estimation windows**. Each window holds the experiment's
10-asset subset, two per sector. All of them are built here with the engine's own slicer, so the diagnostics below see exactly the data the backtest will.

In [2]:
close, volume, extraction_date = data.download_raw()
panel = data.build_panel(close, volume, extraction_date)
prices = panel.prices[cfg.ASSET_ORDER]
schedule = bt.Schedule()
experiments = bt.sample_experiments(prices, schedule)

windows = []   # (experiment, rebalance, returns window, daily risk-free over the window)
for i, e in experiments.iterrows():
    returns = bt.experiment_prices(prices, e).pct_change().iloc[1:]
    for k, execution in enumerate(schedule.execution_positions()):
        formation = execution - schedule.lag
        window = bt.estimation_window(returns, formation,
                                      schedule.initial_lookback if k == 0 else schedule.lookback)
        rf = panel.risk_free.reindex(window.index).ffill() / 100 / cfg.TRADING_DAYS
        windows.append((i, k, window, rf))

print(f"Extraction date : {panel.extraction_date.isoformat()}")
print(f"Windows         : {len(windows)} ({len(experiments)} experiments x "
      f"{len(schedule.execution_positions())} rebalances)")

Extraction date : 2026-10-07
Windows         : 400 (100 experiments x 4 rebalances)


---

## Estimation inputs, shared by all eight

| Input | Estimator | Used by |
| --- | --- | --- |
| μ | Sample mean of daily simple returns × 252 | MV, MSR |
| Σ | **Sample covariance** × 252 | GMV, MV, MSR, ERC, MDP |
| σ | √diag(Σ) | IV, MDP |
| C | Correlation matrix from Σ | MDC |
| r_f | Mean `^IRX` over the window, annualised | MSR |

**Why the sample covariance, not Ledoit–Wolf.** Shrinkage is for when estimation noise swamps the
signal. Here each window has 10 assets and 252–503 observations, about 25 or more per asset, and
notebook 03 found the correlation matrix adequately conditioned (median condition number 18). The
Ledoit–Wolf constant-correlation target has a specific cost for this brief: it pulls every correlation
towards the average, and **when all correlations are equal, ERC and MDP reduce exactly to IV** (test 6
below). So the question is how hard it would shrink, and where:

In [3]:
intensity = pd.Series([st.estimate(w, covariance="ledoit-wolf").shrinkage for _, _, w, _ in windows])
crash = pd.Series([w.index[0] <= pd.Timestamp("2020-03-16") <= w.index[-1] for _, _, w, _ in windows])
print("Ledoit-Wolf intensity across the 400 windows (0 = sample matrix, 1 = constant correlation):")
print(intensity.describe(percentiles=[0.1, 0.5, 0.9]).round(2).to_string())
print()
print(f"Median intensity, windows containing 16 March 2020: {intensity[crash].median():.2f} (n = {crash.sum()}); "
      f"all other windows: {intensity[~crash].median():.2f}")
print(f"Windows shrunk more than halfway: {(intensity > 0.5).sum()}, of which {(intensity[crash] > 0.5).sum()} "
      "contain the crash")

def distance(a, b):
    return 0.5 * (a - b).abs().sum()

def under(covariance, fn, w):
    previous, st.COVARIANCE = st.COVARIANCE, covariance
    try:
        return fn(w)
    finally:
        st.COVARIANCE = previous

contrast = pd.DataFrame({
    cov: {"ERC vs IV, median distance %": np.median([distance(under(cov, st.equal_risk_contribution, w),
                                                              under(cov, st.inverse_volatility, w))
                                                     for _, _, w, _ in windows]) * 100,
          "MDP vs IV, median distance %": np.median([distance(under(cov, st.most_diversified, w),
                                                              under(cov, st.inverse_volatility, w))
                                                     for _, _, w, _ in windows]) * 100}
    for cov in ("sample", "ledoit-wolf")
}).T
display(fmt.style(contrast, {c: lambda v: fmt.percent(v, 1) for c in contrast.columns}))

Ledoit-Wolf intensity across the 400 windows (0 = sample matrix, 1 = constant correlation):
count    400.00
mean       0.17
std        0.12
min        0.03
10%        0.07
50%        0.14
90%        0.31
max        1.00

Median intensity, windows containing 16 March 2020: 0.28 (n = 45); all other windows: 0.13
Windows shrunk more than halfway: 6, of which 5 contain the crash


,"ERC vs IV, median distance %","MDP vs IV, median distance %"
sample,8.7,33.7
ledoit-wolf,7.2,28.6


**Mild on average, erratic in the crash.** In a typical window Ledoit–Wolf would shrink only about
13% of the way to constant correlation, and on average it would leave the IV/ERC/MDP contrast largely
intact (table above). But the intensity is estimated from fourth moments, which the fat tails of
notebook 02 inflate. In the windows that contain March 2020 the median intensity doubles. Of the six
windows it would shrink more than halfway, **five contain the crash, and two are shrunk all the way:
full constant correlation, where ERC and MDP become IV.** The estimator removes correlation structure
precisely when correlations shift most. That is not a good baseline.

The group therefore **kept the sample covariance**. Ledoit–Wolf stays in `src/strategies.py` as a
switch (`COVARIANCE = "ledoit-wolf"`) for a robustness re-run in workstream F.

---

## D10 — Table 3.2, strategy specifications

Every strategy is long-only, fully invested and unlevered, the common baseline of definitions §7. They
differ in objective and in which estimates they consume. The *shortcomings* column comes from the
group's draft (W-4 in the work-division page). It feeds the trade-off discussion in report section 5.

In [4]:
spec = st.specification_table()
print("Table 3.2 - Strategy specifications. All eight are long-only, fully invested and unlevered; "
      "μ and Σ annualised (x 252) from daily simple returns over the estimation window.")
display(spec)

Table 3.2 - Strategy specifications. All eight are long-only, fully invested and unlevered; μ and Σ annualised (x 252) from daily simple returns over the estimation window.


,Objective,Constraints,Parameters,Estimation inputs,Shortcomings
Strategy,,,,,
EW,wᵢ = 1/N,"w ≥ 0, Σw = 1",—,None,Ignores risk and correlation entirely; the risk is dominated by the most volatile asset
GMV,min wᵀΣw,"w ≥ 0, Σw = 1",—,Σ,Ignores expected returns; tends to concentrate in the lowest-volatility asset
MV,max wᵀμ − (λ/2)wᵀΣw,"w ≥ 0, Σw = 1",λ = 3 (annual units),"μ, Σ",Very sensitive to errors in μ; often ends up in a corner holding one or two assets
MSR,max (wᵀμ − r_f) / √(wᵀΣw),"w ≥ 0, Σw = 1",r_f = mean ^IRX over the window,"μ, Σ, r_f","As MV, plus undefined when no asset beats r_f (logged as a failure)"
IV,wᵢ ∝ 1/σᵢ,"w ≥ 0, Σw = 1",—,σ (diagonal of Σ),"Ignores correlations, so two highly correlated assets get full weight each"
ERC,wᵢ(Σw)ᵢ = wᵀΣw / N for all i,"w ≥ 0, Σw = 1",Equal risk budgets 1/N,Σ,Low-risk assets get large weights; the result depends on the quality of the Σ estimate
MDP,max (wᵀσ) / √(wᵀΣw),"w ≥ 0, Σw = 1",—,"σ, Σ",Favours low-correlation assets whatever their return; can concentrate
MDC,"min wᵀCw, C = correlation matrix","w ≥ 0, Σw = 1",—,C,"Treats every asset as equally volatile, so it can overweight a very risky asset that is uncorrelated with the rest"


**How the optimisers are solved.** GMV, MV, MSR, MDP and MDC are all one problem,
*min ½xᵀQx + cᵀx subject to aᵀx = b, x ≥ 0*, with different Q, c and a:

| Strategy | Q | c | a |
| --- | --- | --- | --- |
| GMV | Σ | 0 | 1 |
| MV | λΣ | −μ | 1 |
| MSR | Σ | 0 | μ − r_f, then w = x / Σx |
| MDP | Σ | 0 | σ, then w = x / Σx |
| MDC | C | 0 | 1 |

MSR and MDP maximise a ratio, which is not directly a QP. The substitution *y = w / (aᵀw)* turns each
into the QP above (Cornuéjols and Tütüncü's standard trick). `solve_qp` solves it **exactly**: it
enumerates the possible sets of non-zero weights, 1,023 for a 10-asset subset, and solves the
optimality (KKT) conditions on each, in about 20 ms. There is no iterative solver, so no tolerance to tune and no convergence
failure. The only failure left is a problem with no solution. ERC has no QP form. It is solved by
Newton's method on Spinu's (2013) convex reformulation, and the risk contributions are then checked to
be equal to 1e-8.

---

## D9 — Unit tests

Each test checks a strategy against a case with a known answer, or checks a property that must hold on
every one of the 400 real windows. The `returns_with` helper builds a synthetic return panel whose
sample mean and covariance are **exactly** a chosen μ and Σ. That lets a test state the answer before
calling the strategy.

In [5]:
def returns_with(mu, cov, t=252, seed=0):
    # Daily returns whose sample mean x 252 is exactly `mu` and whose sample covariance x 252 is exactly `cov`.
    mu, cov = np.asarray(mu, float), np.asarray(cov, float)
    n = len(mu)
    z = np.random.default_rng(seed).normal(size=(t, n))
    z -= z.mean(axis=0)
    z = z @ np.linalg.inv(np.linalg.cholesky(np.cov(z, rowvar=False))).T        # sample cov = I
    x = z @ np.linalg.cholesky(cov / cfg.TRADING_DAYS).T + mu / cfg.TRADING_DAYS
    return pd.DataFrame(x, columns=[f"A{i}" for i in range(n)])


def constant_correlation(vol, rho):
    vol = np.asarray(vol, float)
    corr = np.full((len(vol), len(vol)), rho)
    np.fill_diagonal(corr, 1.0)
    return corr * np.outer(vol, vol)


VOL = [0.10, 0.20, 0.30, 0.60]
EXACT = 1e-9


def close_to(w, expected, tol=EXACT):
    np.testing.assert_allclose(np.asarray(w, float), np.asarray(expected, float) / np.sum(expected), atol=tol)


# ---- 1. The exact QP solver agrees with a general-purpose optimiser -------- #
def test_1_qp_matches_slsqp():
    rng = np.random.default_rng(1)
    for _ in range(150):
        n = int(rng.integers(2, cfg.SUBSET_SIZE + 1))             # up to the subset size the backtest uses
        a_ = rng.normal(size=(n, n))
        q, c = a_ @ a_.T + 0.01 * np.eye(n), rng.normal(size=n)
        x = st.solve_qp(q, c, np.ones(n))
        ref = minimize(lambda z: 0.5 * z @ q @ z + c @ z, np.full(n, 1 / n), jac=lambda z: q @ z + c,
                       bounds=[(0, None)] * n, method="SLSQP",
                       constraints=[{"type": "eq", "fun": lambda z: z.sum() - 1}],
                       options={"ftol": 1e-14, "maxiter": 500})
        assert x.min() >= 0 and np.isclose(x.sum(), 1)
        # Never worse than SLSQP, to rounding.
        assert 0.5 * x @ q @ x + c @ x <= ref.fun + 1e-12


# ---- 2. The synthetic panel really has the stated moments ------------------ #
def test_2_returns_with_is_exact():
    cov = constant_correlation(VOL, 0.3)
    est = st.estimate(returns_with([0.05, 0.06, 0.07, 0.08], cov))
    np.testing.assert_allclose(est.cov.to_numpy(), cov, atol=1e-12)
    np.testing.assert_allclose(est.mu.to_numpy(), [0.05, 0.06, 0.07, 0.08], atol=1e-12)


# ---- 3. EW and GMV against closed forms ------------------------------------ #
def test_3_ew_and_gmv():
    r = returns_with(np.zeros(4), np.diag(np.square(VOL)))
    close_to(st.equal_weight(r), np.ones(4))
    close_to(st.global_minimum_variance(r), 1 / np.square(VOL))                 # uncorrelated: w ∝ 1/σ²
    # Correlated, interior solution: w ∝ Σ⁻¹1.
    cov = constant_correlation(VOL[:3], 0.2)
    close_to(st.global_minimum_variance(returns_with(np.zeros(3), cov)), np.linalg.solve(cov, np.ones(3)))


# ---- 4. MV spans the frontier: λ -> ∞ is GMV, λ -> 0 is the top-μ asset ---- #
def test_4_mv_limits():
    r = returns_with([0.02, 0.05, 0.09, 0.15], constant_correlation(VOL, 0.2))
    close_to(st.mean_variance(r, risk_aversion=1e7), st.global_minimum_variance(r), tol=1e-6)
    close_to(st.mean_variance(r, risk_aversion=1e-7), [0, 0, 0, 1])
    # λ = 3 is neither end.
    w = st.mean_variance(r)
    assert (w - st.global_minimum_variance(r)).abs().max() > 0.05 and w.max() < 1


# ---- 5. MSR: tangency formula when interior, raises when undefined --------- #
def test_5_msr():
    mu = np.array([0.06, 0.08, 0.10, 0.14])
    cov = constant_correlation(VOL, 0.2)
    rf = pd.Series(0.02 / cfg.TRADING_DAYS, index=range(252))
    tangency = np.linalg.solve(cov, mu - 0.02)
    assert (tangency > 0).all()                                                # interior case
    close_to(st.maximum_sharpe(returns_with(mu, cov), risk_free=rf), tangency)
    try:
        st.maximum_sharpe(returns_with(np.full(4, 0.01), cov), risk_free=rf)
    except st.InfeasibleError:
        pass
    else:
        raise AssertionError("MSR returned weights when no asset beats the risk-free rate")
    try:
        st.maximum_sharpe(returns_with(mu, cov))
    except ValueError:
        pass
    else:
        raise AssertionError("MSR ran without a risk-free rate")


# ---- 6. IV, ERC and MDP: identical exactly when correlations are equal ----- #
def test_6_iv_erc_mdp_under_constant_correlation():
    for rho in (0.0, 0.3, 0.7):
        r = returns_with(np.zeros(4), constant_correlation(VOL, rho))
        iv = st.inverse_volatility(r)
        close_to(iv, 1 / np.array(VOL))
        close_to(st.equal_risk_contribution(r), iv.to_numpy())
        close_to(st.most_diversified(r), iv.to_numpy())
    # ...and different once correlations differ: assets 0 and 1 highly correlated, 2 and 3 not.
    corr = np.array([[1, .8, 0, 0], [.8, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
    r = returns_with(np.zeros(4), corr * np.outer(VOL, VOL))
    erc, iv = st.equal_risk_contribution(r), st.inverse_volatility(r)
    assert (erc - iv).iloc[:2].lt(0).all() and (erc - iv).iloc[2:].gt(0).all()   # ERC cuts the correlated pair


# ---- 7. MDC: equal weight when uncorrelated, ignores volatility ------------ #
def test_7_mdc():
    close_to(st.maximum_decorrelation(returns_with(np.zeros(4), np.diag(np.square(VOL)))), np.ones(4))
    corr = constant_correlation([1, 1, 1], 0.4)
    a = st.maximum_decorrelation(returns_with(np.zeros(3), corr * np.outer([.1, .2, .3], [.1, .2, .3])))
    b = st.maximum_decorrelation(returns_with(np.zeros(3), corr * np.outer([.5, .1, .9], [.5, .1, .9])))
    close_to(a, b.to_numpy())                                                  # same C, same weights


# ---- 8. ERC equalises risk on every real window ---------------------------- #
def test_8_erc_on_real_windows():
    for _, _, w, _ in windows:
        rc = st.risk_contributions(st.equal_risk_contribution(w), st.estimate(w).cov)
        assert (rc - 1 / len(rc)).abs().max() < 1e-8


# ---- 9. All eight give valid weights on every real window ------------------ #
def test_9_valid_on_all_windows():
    for name, fn in st.STRATEGIES.items():
        for _, _, w, rf in windows:
            try:
                weights = fn(w, risk_free=rf) if name == "MSR" else fn(w)
            except st.InfeasibleError:
                assert name == "MSR", name                                     # only MSR may be undefined
                continue
            assert list(weights.index) == list(w.columns)
            assert weights.min() >= 0 and abs(weights.sum() - 1) < 1e-12, name


# ---- 10. Ledoit-Wolf, kept as a switch, behaves as specified ----------------- #
def test_10_ledoit_wolf():
    for _, _, w, _ in windows[:40]:
        est, sample = st.estimate(w, covariance="ledoit-wolf"), st.estimate(w)
        assert 0 <= est.shrinkage <= 1
        np.testing.assert_allclose(np.diag(est.cov), np.diag(sample.cov), rtol=1e-12)   # variances kept
        assert np.linalg.eigvalsh(est.cov.to_numpy()).min() > 0


TESTS = [test_1_qp_matches_slsqp, test_2_returns_with_is_exact, test_3_ew_and_gmv, test_4_mv_limits,
         test_5_msr, test_6_iv_erc_mdp_under_constant_correlation, test_7_mdc,
         test_8_erc_on_real_windows, test_9_valid_on_all_windows, test_10_ledoit_wolf]

In [6]:
outcomes = []
for test in TESTS:
    try:
        test()
        outcomes.append((test.__name__, "PASS", ""))
    except Exception as exc:
        outcomes.append((test.__name__, "FAIL", f"{type(exc).__name__}: {exc}"))

test_report = pd.DataFrame(outcomes, columns=["Test", "Result", "Detail"]).set_index("Test")
display(test_report)
assert (test_report["Result"] == "PASS").all(), "unit tests failed"
print(f"All {len(TESTS)} tests pass.")

,Result,Detail
Test,,
test_1_qp_matches_slsqp,PASS,
test_2_returns_with_is_exact,PASS,
test_3_ew_and_gmv,PASS,
test_4_mv_limits,PASS,
test_5_msr,PASS,
test_6_iv_erc_mdp_under_constant_correlation,PASS,
test_7_mdc,PASS,
test_8_erc_on_real_windows,PASS,
test_9_valid_on_all_windows,PASS,


All 10 tests pass.


---

## D11 — Does IV differ from ERC?

IV sizes each asset by its own volatility alone. ERC sizes it by its contribution to *portfolio* risk,
which also depends on how it co-moves with the others. The two coincide exactly when every pair has the
same correlation (test 6). They separate as correlations become unequal: ERC gives **less** weight than
IV to an asset that is highly correlated with the rest, and **more** to one that diversifies.

**One fixed window.** Experiment 1's opening estimate, taken as the first window drawn rather than
picked for effect:

In [7]:
FIXED = 0                                         # experiment 1, rebalance 0
exp_id, _, fixed, fixed_rf = windows[FIXED]
est = st.estimate(fixed)
iv, erc = st.inverse_volatility(fixed), st.equal_risk_contribution(fixed)

corr = est.corr
avg_corr = (corr.sum() - 1) / (len(corr) - 1)
table_3_3 = pd.DataFrame({
    "Volatility %": est.vol * 100,
    "Avg. correlation with others": avg_corr,
    "IV weight %": iv * 100,
    "ERC weight %": erc * 100,
    "ERC − IV, pp": (erc - iv) * 100,
    "IV risk share %": st.risk_contributions(iv, est.cov) * 100,
    "ERC risk share %": st.risk_contributions(erc, est.cov) * 100,
})
table_3_3.index.name = "Ticker"

print(f"Table 3.3 - IV against ERC on one estimation window: experiment {exp_id}, "
      f"{fixed.index[0].date()} to {fixed.index[-1].date()}, n = {len(fixed)} daily returns. "
      "Risk share = each asset's share of portfolio variance.")
display(fmt.style(table_3_3, {
    "Volatility %": lambda v: fmt.percent(v, 1),
    "Avg. correlation with others": lambda v: fmt.number(v, 2),
    "IV weight %": lambda v: fmt.percent(v, 1),
    "ERC weight %": lambda v: fmt.percent(v, 1),
    "ERC − IV, pp": lambda v: fmt.number(v, 1),
    "IV risk share %": lambda v: fmt.percent(v, 1),
    "ERC risk share %": lambda v: fmt.percent(v, 1),
}))
print(corr.round(2).to_string())

Table 3.3 - IV against ERC on one estimation window: experiment 1, 2016-03-31 to 2018-03-28, n = 503 daily returns. Risk share = each asset's share of portfolio variance.


,Volatility %,Avg. correlation with others,IV weight %,ERC weight %,"ERC − IV, pp",IV risk share %,ERC risk share %
Ticker,,,,,,,
AAPL,20.1,0.21,8.4,7.3,-1.1,11.5,10.0
AMZN,23.3,0.19,7.3,6.6,-0.7,10.7,10.0
PFE,16.3,0.21,10.4,9.1,-1.3,11.5,10.0
UNH,17.3,0.22,9.8,8.3,-1.5,11.9,10.0
BAC,24.9,0.17,6.8,7.2,0.4,10.0,10.0
GS,22.9,0.20,7.4,6.8,-0.6,11.2,10.0
KO,12.4,0.21,13.7,11.6,-2.1,11.7,10.0
PEP,12.1,0.20,13.9,12.4,-1.5,11.1,10.0
GLD,12.1,-0.03,14.0,22.2,8.2,3.1,10.0


Ticker  AAPL  AMZN   PFE   UNH   BAC    GS    KO   PEP   GLD   SLV
Ticker                                                            
AAPL    1.00  0.43  0.22  0.28  0.29  0.32  0.18  0.23 -0.10  0.02
AMZN    0.43  1.00  0.14  0.23  0.21  0.21  0.23  0.20 -0.00  0.04
PFE     0.22  0.14  1.00  0.41  0.35  0.36  0.22  0.21 -0.10  0.05
UNH     0.28  0.23  0.41  1.00  0.37  0.36  0.25  0.19 -0.09 -0.01
BAC     0.29  0.21  0.35  0.37  1.00  0.82  0.07  0.00 -0.43 -0.18
GS      0.32  0.21  0.36  0.36  0.82  1.00  0.13  0.07 -0.35 -0.10
KO      0.18  0.23  0.22  0.25  0.07  0.13  1.00  0.74  0.03  0.08
PEP     0.23  0.20  0.21  0.19  0.00  0.07  0.74  1.00  0.03  0.10
GLD    -0.10 -0.00 -0.10 -0.09 -0.43 -0.35  0.03  0.03  1.00  0.79
SLV     0.02  0.04  0.05 -0.01 -0.18 -0.10  0.08  0.10  0.79  1.00


**Across all 400 windows.** If the difference tracks the correlation structure, an asset's ERC − IV
weight gap should fall as its average correlation with the other three rises. The gap and the
correlation are both measured relative to the window's own average, so that windows with different
correlation levels are comparable:

In [8]:
rows = []
for i, k, w, _ in windows:
    est_w = st.estimate(w)
    c = est_w.corr
    avg = (c.sum() - 1) / (len(c) - 1)
    gap = st.equal_risk_contribution(w) - st.inverse_volatility(w)
    for asset in w.columns:
        rows.append({"experiment": i, "rebalance": k, "asset": asset, "gap_pp": gap[asset] * 100,
                     "avg_corr_rel": avg[asset] - avg.mean(),
                     "corr_dispersion": c.where(~np.eye(len(c), dtype=bool)).stack().std()})
gaps = pd.DataFrame(rows)
per_window = gaps.groupby(["experiment", "rebalance"]).agg(max_gap=("gap_pp", lambda g: g.abs().max()),
                                                           dispersion=("corr_dispersion", "first"))

slope_corr = gaps["gap_pp"].corr(gaps["avg_corr_rel"])
size_corr = per_window["max_gap"].corr(per_window["dispersion"])
iv_erc_summary = pd.DataFrame({
    "Value": [per_window["max_gap"].median(), per_window["max_gap"].quantile(0.9), per_window["max_gap"].max(),
              slope_corr, size_corr],
}, index=["Largest |ERC − IV| per window, median (pp)", "... 90th percentile (pp)", "... maximum (pp)",
          "Corr(ERC − IV gap, asset's relative avg. correlation)",
          "Corr(largest gap, dispersion of pairwise correlations)"])
display(fmt.style(iv_erc_summary, {"Value": lambda v: fmt.number(v, 2)}))
assert slope_corr < -0.5 and size_corr > 0.3

,Value
"Largest |ERC − IV| per window, median (pp)",7.30
... 90th percentile (pp),18.97
... maximum (pp),31.86
"Corr(ERC − IV gap, asset's relative avg. correlation)",-0.94
"Corr(largest gap, dispersion of pairwise correlations)",0.88


**Reading.** IV and ERC differ in every window, in the direction theory predicts, and by a material
amount. The more correlated an asset is with the rest, the more ERC cuts it relative to IV (a
correlation of about −0.9 in the first diagnostic). The windows where correlations are most unequal
produce the largest gaps (about +0.9 in the second). In a typical window the largest single-asset gap
is about 7 percentage points, and in one window in ten it reaches nearly 20.

The fixed window shows why. IV gives `GLD` 14%, judged on its volatility alone. ERC gives it 22%,
because gold is *negatively* correlated with the two banks in the subset (−0.43 and −0.35), so it
diversifies far more than its volatility suggests. Under IV, gold carries only 3% of portfolio risk,
against the 10% ERC's equal budgets require. The other assets, each correlated with several of the
rest, are trimmed by a point or two.

The contrast is much larger than a weakly and evenly correlated universe would give, and it comes from
the sector structure of Figure 1.1: tight blocks within sectors, and diversifiers (gold, Treasuries)
outside them. The report should say exactly that. The IV/ERC difference is driven by correlation, and
this universe has enough correlation structure to make it large.

---

## D12 — Are MV, GMV and MSR three different portfolios?

All three sit on the same long-only efficient frontier. GMV is its minimum-risk end. MSR is the point
with the highest Sharpe ratio. MV is the point the risk-aversion coefficient λ = 3 selects. The pre-set
λ is what makes MV a separate portfolio. On the fixed window:

In [9]:
gmv, mv, msr = (st.global_minimum_variance(fixed), st.mean_variance(fixed),
                st.maximum_sharpe(fixed, risk_free=fixed_rf))
table_3_4 = pd.DataFrame({"μ %/yr": est.mu * 100, "σ %/yr": est.vol * 100,
                          "GMV %": gmv * 100, "MV (λ = 3) %": mv * 100, "MSR %": msr * 100})
table_3_4.index.name = "Ticker"

def portfolio_stats(w):
    return pd.Series({"μ %/yr": w @ est.mu * 100, "σ %/yr": np.sqrt(w @ est.cov @ w) * 100})

for name, w in (("GMV", gmv), ("MV (λ = 3)", mv), ("MSR", msr)):
    s = portfolio_stats(w)
    print(f"{name:<11} ex-ante μ {s['μ %/yr']:6.1f}%  σ {s['σ %/yr']:5.1f}%")

rf_annual = fixed_rf.mean() * cfg.TRADING_DAYS * 100
print(f"\nTable 3.4 - MV, GMV and MSR weights on one estimation window: experiment {exp_id}, "
      f"{fixed.index[0].date()} to {fixed.index[-1].date()}, n = {len(fixed)}; r_f = {rf_annual:.2f}%/yr. "
      "μ and σ are in-sample estimates, not results.")
display(fmt.style(table_3_4, {c: lambda v: fmt.percent(v, 1) for c in table_3_4.columns}))

GMV         ex-ante μ   11.7%  σ   6.8%
MV (λ = 3)  ex-ante μ   45.4%  σ  18.9%
MSR         ex-ante μ   28.5%  σ  10.3%

Table 3.4 - MV, GMV and MSR weights on one estimation window: experiment 1, 2016-03-31 to 2018-03-28, n = 503; r_f = 0.76%/yr. μ and σ are in-sample estimates, not results.


,μ %/yr,σ %/yr,GMV %,MV (λ = 3) %,MSR %
Ticker,,,,,
AAPL,24.8,20.1,3.6,0.0,0.0
AMZN,46.4,23.3,0.1,63.8,23.9
PFE,13.1,16.3,6.7,0.0,0.0
UNH,29.2,17.3,3.5,0.0,18.5
BAC,43.8,24.9,12.2,36.2,24.2
GS,27.3,22.9,1.3,0.0,0.0
KO,0.4,12.4,9.4,0.0,0.0
PEP,6.7,12.1,18.6,0.0,0.0
GLD,4.3,12.1,44.6,0.0,33.4


**Across all 400 windows.** The distance between two weight vectors is measured as **½ Σ|wᵢ − vᵢ|**:
the share of the portfolio that would have to change hands to turn one into the other. Two portfolios
**coincide** when that is below 0.01%.

In [10]:
def distance(a, b):
    return 0.5 * (a - b).abs().sum()

rows, msr_weights = [], []
for i, k, w, rf in windows:
    g, m = st.global_minimum_variance(w), st.mean_variance(w)
    try:
        s = st.maximum_sharpe(w, risk_free=rf)
    except st.InfeasibleError:
        s = None
    msr_weights.append(s)
    rows.append({"experiment": i, "rebalance": k,
                 "GMV–MV": distance(g, m),
                 "GMV–MSR": np.nan if s is None else distance(g, s),
                 "MV–MSR": np.nan if s is None else distance(m, s),
                 "mv_single": m.max() > 1 - 1e-9,
                 "msr_single": False if s is None else s.max() > 1 - 1e-9,
                 "mv_asset": m.idxmax()})
d12 = pd.DataFrame(rows)

COINCIDE = 1e-4
pairs = ["GMV–MV", "GMV–MSR", "MV–MSR"]
table_d12 = pd.DataFrame({
    "Windows compared": [d12[p].notna().sum() for p in pairs],
    "Median distance %": [d12[p].median() * 100 for p in pairs],
    "10th pct. distance %": [d12[p].quantile(0.1) * 100 for p in pairs],
    "Coincide": [(d12[p] < COINCIDE).sum() for p in pairs],
}, index=pd.Index(pairs, name="Pair"))
print("Distinctness of GMV, MV (λ = 3) and MSR across the 400 estimation windows. "
      f"MSR undefined (no asset above r_f) in {d12['GMV–MSR'].isna().sum()} windows.")
display(fmt.style(table_d12, {"Windows compared": fmt.count, "Coincide": fmt.count,
                              "Median distance %": lambda v: fmt.percent(v, 1),
                              "10th pct. distance %": lambda v: fmt.percent(v, 1)}))

same = d12[d12["MV–MSR"] < COINCIDE]
print(f"MV = MSR in {len(same)} windows; in {int((same['mv_single'] & same['msr_single']).sum())} of them "
      "both hold 100% of one asset.")
print("That asset:", same["mv_asset"].value_counts().to_dict())
print(f"MV holds 100% of one asset in {d12['mv_single'].mean() * 100:.1f}% of all windows.")

Distinctness of GMV, MV (λ = 3) and MSR across the 400 estimation windows. MSR undefined (no asset above r_f) in 0 windows.


,Windows compared,Median distance %,10th pct. distance %,Coincide
Pair,,,,
GMV–MV,400,91.1,68.5,0
GMV–MSR,400,59.2,32.4,0
MV–MSR,400,46.2,11.6,7


MV = MSR in 7 windows; in 7 of them both hold 100% of one asset.
That asset: {'ORCL': 2, 'AAPL': 1, 'MSFT': 1, 'JPM': 1, 'KO': 1, 'PEP': 1}
MV holds 100% of one asset in 30.5% of all windows.


**Context for the λ decision: the same count at other values of λ.** This is a weight diagnostic only.
No performance figure has been computed for any λ.

In [11]:
lam_rows = []
for lam in (1, 2, 3, 5, 10, 20):
    mv_l = [st.mean_variance(w, risk_aversion=lam) for _, _, w, _ in windows]
    coincide = sum(s is not None and distance(m_, s) < COINCIDE for m_, s in zip(mv_l, msr_weights))
    lam_rows.append({"λ": lam, "MV = MSR (windows)": coincide,
                     "MV all in one asset %": np.mean([m_.max() > 1 - 1e-9 for m_ in mv_l]) * 100,
                     "Median distance to GMV %": np.median([distance(m_, st.global_minimum_variance(w))
                                                            for m_, (_, _, w, _) in zip(mv_l, windows)]) * 100})
lam_table = pd.DataFrame(lam_rows).set_index("λ")
display(fmt.style(lam_table, {"MV = MSR (windows)": fmt.count,
                              "MV all in one asset %": lambda v: fmt.percent(v, 1),
                              "Median distance to GMV %": lambda v: fmt.percent(v, 1)}))

,MV = MSR (windows),MV all in one asset %,Median distance to GMV %
λ,,,
1,7,73.2,96.2
2,7,50.2,94.1
3,7,30.5,91.1
5,4,8.5,84.4
10,0,0.0,69.7
20,0,0.0,50.2


**Reading, and the λ decision.** The three are distinct portfolios. MV is never close to GMV (median
distance 91%), GMV never coincides with MSR, and **MV and MSR coincide in only 7 of the 400 windows,
every one of them a corner**: both put 100% in the same single stock, the one with an extreme trailing
return. On a corner the risk penalty in MV is too weak to pull the portfolio off the top-return asset,
and the same asset also has the best Sharpe ratio.

The fixed window shows the three personalities. **GMV** spreads across nine of ten assets, led by gold
at 45%, the lowest-volatility, least-correlated asset. **MSR** holds four, and **MV** only two
(`AMZN` and `BAC`, the two highest trailing returns). That concentration is the general pattern: at
λ = 3 MV holds **a single asset in 30% of windows**. It is the estimation-error sensitivity listed under
*Shortcomings* in Table 3.2. A one-year trailing mean is a noisy estimate, and MV acts on it at face
value.

The group **kept λ = 3**, the value fixed before any run (definitions §7). The table above shows that a
higher λ would spread MV out (λ = 10 removes the corners). Choosing it after looking at these weights
would be tuning, though, and λ = 3 already meets the brief's requirement that MV differ from GMV and
MSR. Report section 4 should state:

- MV is distinct from GMV in every window and from MSR in all but 7;
- where MV and MSR coincide, it is at a single-asset corner;
- MV's concentration (a single asset in 30% of windows) is a symptom of estimation error in μ, not
  of a mis-specified objective.

---

## Pre-flight through the engine

Every strategy runs once through `run_backtest` on all 100 experiments. This checks that the signatures
plug into the loop and shows what the failure log (Table A.1) will contain. **Only failures are
inspected here.** Returns are discarded and reported in notebook 05.

In [12]:
runs = []
for name, fn in st.STRATEGIES.items():
    for i, e in experiments.iterrows():
        runs.append(bt.run_backtest(bt.experiment_prices(prices, e), fn, risk_free=panel.risk_free,
                                    experiment=i, strategy=name))
failures = bt.failure_log(runs)

print(f"Runs completed: {len(runs)} of {len(st.STRATEGIES) * len(experiments)}; none dropped.")
counts = failures.groupby("strategy").size().reindex(cfg.STRATEGY_ORDER, fill_value=0)
print("Optimisation failures by strategy:", counts.to_dict())
if failures.empty:
    print("Failure log: empty. Every strategy produced valid weights at every one of the 400 rebalances.")
else:
    display(failures.assign(date=pd.to_datetime(failures["date"]).dt.date))

Runs completed: 800 of 800; none dropped.
Optimisation failures by strategy: {'EW': 0, 'GMV': 0, 'MV': 0, 'MSR': 0, 'IV': 0, 'ERC': 0, 'MDP': 0, 'MDC': 0}
Failure log: empty. Every strategy produced valid weights at every one of the 400 rebalances.


**The failure log is empty.** In every one of the 400 windows at least one asset's trailing mean beat
the T-bill rate, so MSR was always defined, and every other strategy always produced valid weights.
Table A.1 will therefore report *no failures*, explicitly, rather than being omitted (plan E10). The
handling stays in place. A later universe or period in which every asset in a subset lags cash would
still be logged and held, never dropped.

---

## Done-when checks

In [13]:
checks = []
checks.append(("D9 unit tests pass", (test_report["Result"] == "PASS").all()))
checks.append(("D9 eight functions, fixed order, interchangeable in the loop",
               list(st.STRATEGIES) == cfg.STRATEGY_ORDER and len(runs) == 8 * len(experiments)))
checks.append(("D10 Table 3.2 covers all eight, with a shortcomings column",
               list(spec.index) == cfg.STRATEGY_ORDER and "Shortcomings" in spec.columns))
checks.append(("D11 IV and ERC differ on the fixed window and the gap tracks correlation",
               (iv - erc).abs().max() > 1e-4 and slope_corr < -0.5))
checks.append(("D12 MV, GMV and MSR are three distinct vectors on the fixed window",
               min(distance(gmv, mv), distance(gmv, msr), distance(mv, msr)) > COINCIDE))
checks.append(("D12 every MV = MSR coincidence is a single-asset corner",
               bool((same["mv_single"] & same["msr_single"]).all())))
checks.append(("Only MSR fails, and only for want of an asset above r_f",
               set(failures["strategy"]) <= {"MSR"} and failures["error"].str.contains("risk-free").all()))

for label, ok in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), "workstream D exit criteria not met"

[PASS] D9 unit tests pass
[PASS] D9 eight functions, fixed order, interchangeable in the loop
[PASS] D10 Table 3.2 covers all eight, with a shortcomings column
[PASS] D11 IV and ERC differ on the fixed window and the gap tracks correlation
[PASS] D12 MV, GMV and MSR are three distinct vectors on the fixed window
[PASS] D12 every MV = MSR coincidence is a single-asset corner
[PASS] Only MSR fails, and only for want of an asset above r_f


---

## Handoff

Notebook 05 imports the strategies and runs the full grid:

```python
from src import backtest as bt, strategies as st

for name, fn in st.STRATEGIES.items():          # fixed strategy order
    ...  bt.run_backtest(block, fn, costs, risk_free=panel.risk_free, experiment=i, strategy=name)
```

| Item | Where it goes |
| --- | --- |
| Table 3.2 | Report section 4 |
| Tables 3.3–3.4 and the D11/D12 summaries | Section 4, one paragraph each; full tables in the appendix |
| MSR failures | Table A.1, via `bt.failure_log` in notebook 05 |
| λ = 3 kept; coincidences at corners | Section 4, stated explicitly |
| Ledoit–Wolf switch | Optional robustness row in workstream F |